# Silent Failure Project — Environment Setup & EDA

Verifies:
1. All dependencies from `requirements.txt` are importable
2. Both UCI datasets load correctly (Pima Diabetes + Cleveland Heart Disease)
3. EDA plots are produced for each dataset

In [ ]:
# ── 1. Dependency smoke-test ────────────────────────────────────────────────
import importlib, sys

required = [
    'xgboost', 'ngboost', 'mapie', 'torch',
    'sklearn', 'pandas', 'numpy', 'scipy',
    'matplotlib', 'seaborn', 'streamlit', 'ucimlrepo'
]

for pkg in required:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, '__version__', 'n/a')
    print(f'  {pkg:<15} {ver}')

print('\nAll dependencies OK.')

In [ ]:
# ── 2. Load datasets ────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from src.data_loader import load_pima, load_cleveland

X_pima, y_pima = load_pima()
X_clev, y_clev = load_cleveland()

print(f'Pima      — features: {X_pima.shape}, labels: {y_pima.shape}, '
      f'positive rate: {y_pima.mean():.1%}')
print(f'Cleveland — features: {X_clev.shape}, labels: {y_clev.shape}, '
      f'positive rate: {y_clev.mean():.1%}')

In [ ]:
# ── 3. EDA — Pima Diabetes ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

df_pima = X_pima.copy()
df_pima['target'] = y_pima

features_pima = list(X_pima.columns)
n = len(features_pima)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
fig.suptitle('Pima Diabetes — Feature Distributions by Class', fontsize=14, y=1.01)

for ax, feat in zip(axes.flat, features_pima):
    for cls, label in [(0, 'Negative'), (1, 'Positive')]:
        vals = df_pima.loc[df_pima['target'] == cls, feat].dropna()
        ax.hist(vals, bins=25, alpha=0.6, label=label, density=True)
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

# hide unused axes
for ax in axes.flat[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Pima — correlation heatmap
fig, ax = plt.subplots(figsize=(7, 6))
corr = df_pima.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Pima — Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Pima — class balance
fig, ax = plt.subplots(figsize=(4, 3))
counts = pd.Series(y_pima).value_counts().sort_index()
ax.bar(['Negative', 'Positive'], counts.values, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Pima — Class Balance')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. EDA — Cleveland Heart Disease ──────────────────────────────────────
df_clev = X_clev.copy()
df_clev['target'] = y_clev

features_clev = list(X_clev.columns)
n2 = len(features_clev)
nrows2 = (n2 + ncols - 1) // ncols

fig, axes = plt.subplots(nrows2, ncols, figsize=(16, 3.5 * nrows2))
fig.suptitle('Cleveland Heart Disease — Feature Distributions by Class', fontsize=14, y=1.01)

for ax, feat in zip(axes.flat, features_clev):
    for cls, label in [(0, 'No disease'), (1, 'Disease')]:
        vals = df_clev.loc[df_clev['target'] == cls, feat].dropna()
        ax.hist(vals, bins=25, alpha=0.6, label=label, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

for ax in axes.flat[n2:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Cleveland — correlation heatmap
fig, ax = plt.subplots(figsize=(9, 8))
corr2 = df_clev.corr()
mask2 = np.triu(np.ones_like(corr2, dtype=bool))
sns.heatmap(corr2, mask=mask2, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Cleveland — Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Cleveland — class balance
fig, ax = plt.subplots(figsize=(4, 3))
counts2 = pd.Series(y_clev).value_counts().sort_index()
ax.bar(['No disease', 'Disease'], counts2.values, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Cleveland — Class Balance')
ax.set_ylabel('Count')
for i, v in enumerate(counts2.values):
    ax.text(i, v + 2, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Verify train/cal/test splits ────────────────────────────────────────
from src.data_loader import load_dataset

for name in ['pima', 'cleveland']:
    splits = load_dataset(name)
    X_tr, X_cal, X_te, y_tr, y_cal, y_te, scaler = splits
    total = X_tr.shape[0] + X_cal.shape[0] + X_te.shape[0]
    print(f'{name.capitalize():10} | train={X_tr.shape[0]:>3}  cal={X_cal.shape[0]:>3}  '
          f'test={X_te.shape[0]:>3}  total={total}')

print('\nEnvironment verified successfully.')